In [ ]:
from IPython.display import Video, display
import cv2
import numpy as np

from utils.color import detect_color
from utils.ROI import roi, get_overlap_componant

In [16]:
#in_path  = "../data/Video1fixed.mp4"       
in_path  = "../data/Video2dynamique.mp4"  
out_path = "../outputs/group_11_fixed_test.mp4"

cap = cv2.VideoCapture(in_path)
if not cap.isOpened():
    raise IOError(f"Could not open {in_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v") 
writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

ok, first_frame = cap.read()
if not ok:
    print("no frames")

ball_mask = np.zeros_like(first_frame[:,:,1])
mushroom_mask = np.zeros_like(first_frame[:,:,1])
ice_tea_mask = np.zeros_like(first_frame[:,:,1])

while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    # Get the green mask
    green = [0,255,0]
    green_mask = detect_color(frame_bgr, green, [40,40], [255,255], tuning=35)

    # Cloak mask could take several cloaks
    cloak_mask = np.zeros_like(mushroom_mask)
    is_overlapping = False

    # ---- ball ---- #

    blue = [0,0,255]
    blue_mask = detect_color(frame_bgr, blue, [100,100], [255,255], tuning=25)
    ok, roi_mask = roi(blue_mask, s_erod=10, s_dil=1)

    if ok:
        ball_mask = roi_mask

    is_overlapping_ball, overlap_mask = get_overlap_componant(green_mask, ball_mask)

    if is_overlapping_ball:
        cloak_mask = cv2.add(cloak_mask, overlap_mask)
        is_overlapping = True
    
    # ---- mushroom ---- #

    red = [255,0,10]
    red_mask = detect_color(frame_bgr, red, [50,50], [255,255], tuning=30)
    ok, roi_mask = roi(red_mask, s_erod=10, s_dil=1)

    if ok:
        mushroom_mask = roi_mask

    is_overlapping_mushroom, overlap_mask = get_overlap_componant(green_mask, mushroom_mask)

    if is_overlapping_mushroom:
        cloak_mask = cv2.add(cloak_mask, overlap_mask)
        is_overlapping = True

    # ---- ice_tea ---- #

    # Don't detect well

    """
    yellow = [255,255,0]
    yellow_mask = detect_color(frame_bgr, yellow, [150,20], [250,150], tuning=15)
    ok, roi_mask = roi(yellow_mask, s_erod=10, s_dil=11)

    if ok:
        ice_tea_mask = roi_mask

    is_overlapping_ice_tea, overlap_mask = get_overlap_componant(green_mask, ice_tea_mask)

    if is_overlapping_ice_tea:
        cloak_mask = cv2.add(cloak_mask, overlap_mask)
        is_overlapping = True
    """

    # ---- create last frame ---- #

    if is_overlapping:

        s = 3
        SE= cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(s,s))
        cloak_mask = cv2.dilate(cloak_mask,SE)

        visible_environment_mask = cv2.bitwise_not(cloak_mask)

        invisible_cloak = cv2.bitwise_and(first_frame, first_frame, mask=cloak_mask)
        visible_environment = cv2.bitwise_and(frame_bgr, frame_bgr, mask=visible_environment_mask)

        output_frame = cv2.add(visible_environment, invisible_cloak)
    else:
        output_frame = frame_bgr

    # masking = yellow_mask
    # frame = cv2.bitwise_and(frame_bgr, frame_bgr, mask=yellow_mask)
    # writer.write(frame)

    writer.write(output_frame)

cap.release()
writer.release()
print("Saved:", out_path)

display(Video(out_path))

Saved: ../outputs/group_11_fixed_test.mp4
